# Extra deliverable - INF201 

Submission deadline: 4th Dec 2026

### Hand-in information:
* This deliverable should be submitted individually
* Submit Python scripts (.py) and/or a Jupyter notebook (ipynb) as well as a pdf of the same notebook
* Include a link to your GitHub repository and a log of the commits
    * To generate the log, paste this into the terminal after your last commit:<br>
    `git log --graph --pretty=format:"%h - %an, %ar : %s" > git_log.txt`
    * The repository should contain a `requirements.txt` with information on your virtual environment
    * Note: Remember to create a new virtual environment for each deliverable
* Include a discussion of GenAI use, for example a summary of the chat log as you have
done earlier
    * This can be a separate file or be included in the Jupyter notebook
* We do not require a video for this assignment

## Modelling mass-spring oscillations
In this task you will use `sympy` and `scipy` to model a classic situation you might remember from physics class:<br> the oscillations of a mass fastened to a spring.

#### Background
An undamped mass-spring system obeys Newton's second law combined with Hooke's law:$$m\ddot{x} + kx = 0$$where $x(t)$ is the displacement from equilibrium (in meters $\text{m}$), $\ddot{x} = \frac{d^2x}{dt^2}$ is acceleration (in $\text{m/s}^2$), $m$ is mass (in $\text{kg}$), and $k$ is spring stiffness (in $\text{N/m}$). Because there is no friction or damping term ($\dot{x}$), the mass oscillates indefinitely at its natural angular frequency $\omega_0 = \sqrt{k/m}$ (in $\text{rad/s}$).

In this problem, you will find the exact analytical motion of a $2\text{ kg}$ mass on a $200\text{ N/m}$ spring ($2\ddot{x} + 200x = 0$) using `sympy`, then use `scipy` to diagnose and filter out electrical interference from a noisy position sensor.

#### Exact symbolic solution (`sympy`)
* Define `t` as a positive real symbol and `x` as an undefined function `Function('x')(t)`.
* Use `sympy.dsolve` to solve $2\ddot{x} + 200x = 0$ subject to initial displacement $x(0) = 1.5\text{ m}$ <br>and initial velocity $\dot{x}(0) = 0\text{ m/s}$.<br>
* Print the resulting expression for $x(t)$.
* Convert your symbolic solution into a vectorized `NumPy` function `x_exact_fn` using `sympy.lambdify`.

#### Frequency-domain diagnosis (`scipy.fft`)
* Generate the sensor data using the `generate_data.py` script: <br>
`t_num, raw_signal, clean_signal, fs = simulate_sensor(x_exact_fn)` <br>
* Compute the single-sided amplitude spectrum of `raw_signal` using `scipy.fft.rfft`<br>
and `scipy.fft.rfftfreq`, scaling magnitudes by $2/N$ (where $N$ is the number of samples).
* Identify and print the two dominant frequencies (in $\text{Hz}$) in the spectrum. Verify that one matches the oscillator's natural frequency $f_0 = \frac{\omega_0}{2\pi}$ and the other corresponds to the electrical interference.

#### Low-pass filtering and error reduction (`scipy.signal`)
* Design a 4th-order Butterworth low-pass filter (`scipy.signal.butter`) with a cutoff frequency of $10\text{ Hz}$ (`fs=200.0`) to preserve the oscillator signal while removing the $60\text{ Hz}$ interference.
* Apply zero-phase filtering (`scipy.signal.filtfilt`) to `raw_signal` to obtain `recovered_signal`.
* Compute and print the Root Mean Square Error ($\text{RMSE} = \sqrt{\frac{1}{N}\sum (y - y_{\text{clean}})^2}$) both before filtering (`raw_signal` vs. `clean_signal`) and after filtering (`recovered_signal` vs. `clean_signal`).

#### Visualization
* Create a figure with three subplots:
    * Left: Raw time-series (before filtering) 
    * Middle: The single-sided FFT amplitude spectrum ($0$ to $100\text{ Hz}$), marking your $10\text{ Hz}$ filter cutoff with a vertical dashed line. This plot should hav frequency on the x-axis and amplitude on the y-axis.
    * Right: Time-domain comparison over $t \in [0, 2]\text{ s}$ showing `raw_signal`, `recovered_signal`, and `clean_signal`.